<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The **random  forest** method since it can capture relationships between the several signals such as impressions, CTR, ect, without requiring us to assume they are linear

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The **grouped by client** split so that pages from the same client do not appear in both training and test sets. This gives a more realistic estimate of how the model would perform on a new client

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [41]:
# Load dataset
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")

In [42]:
# Defining the 'is_declining_label'
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)  # Where 1=Declining

In [43]:
target = "is_declining_label"

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

X = df[features].copy()
y = df[target].copy()

print("Features:", features)
print("Target:", target)
print("Positive rate:", y.mean())

Features: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']
Target: is_declining_label
Positive rate: 0.5420666666666667


In [44]:
# Create the Split
from sklearn.model_selection import GroupShuffleSplit
groups = df["client_id"]
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [45]:
# Missing values (case)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [46]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [47]:
# Check the Scores
model_scores = model.predict_proba(X_test)[:, 1]

In [48]:
# Creating Precision@50
def precision_at_k(y_true, scores, k=50):
    scores = pd.Series(scores, index=y_true.index)
    top_k = scores.sort_values(ascending=False).head(k).index
    return y_true.loc[top_k].mean()

model_precision_50 = precision_at_k(
    y_test,
    model_scores,
    k=50
)

print(f"Random Forest Precision@50: {model_precision_50:.3f}")

Random Forest Precision@50: 0.680


In [49]:
# Compare from Week 4
import numpy as np

test_data = df.iloc[test_idx].copy()

test_data["ctr_calc"] = (
    test_data["clicks_90d"] /
    test_data["impressions_90d"]
).fillna(0)

max_impressions = df.iloc[train_idx]["impressions_90d"].max()
max_ctr = (
    df.iloc[train_idx]["clicks_90d"] /
    df.iloc[train_idx]["impressions_90d"]
).replace([np.inf, -np.inf], np.nan).fillna(0).max()

test_data["visibility_score"] = (
    test_data["impressions_90d"] / max_impressions
)

test_data["ctr_opportunity"] = (
    1 - test_data["ctr_calc"] / max_ctr
)

test_data["baseline_score"] = (
    test_data["visibility_score"] *
    test_data["ctr_opportunity"]
)

baseline_precision_50 = precision_at_k(
    test_data[target],
    test_data["baseline_score"],
    k=50
)

print(f"Week 4 Baseline Precision@50: {baseline_precision_50:.3f}")

Week 4 Baseline Precision@50: 0.440


In [50]:
# Table to compare
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": [
        "Base rate",
        "Week 4 Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        base_rate,
        baseline_precision_50,
        model_precision_50
    ]
})

comparison

,Method,Precision@50
0,Base rate,0.510952
1,Week 4 Baseline,0.440000
2,Random Forest,0.680000


The **Random Forest** achieved a Precision@50 of 0.68, compared with 0.44 for the **Week 4 Baseline**. This means 68% of the model's top 50 pages were labeled as declining, compared with 44% for the baseline. The model therefore performed better than the simpler rule on the same test set and metric

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [51]:
# Features the Random Forest relied on most
feature_importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

feature_importance

,0
impressions_90d,0.322006
avg_position,0.290758
content_age_days,0.207285
ctr,0.133049
days_since_last_update,0.046903


In [52]:
# Some wrong predictions
predictions = (model_scores >= 0.5).astype(int)

errors = pd.DataFrame({
    "actual": y_test,
    "predicted": predictions,
    "probability": model_scores
})

errors = errors[errors["actual"] != errors["predicted"]]

print("Number of errors:", len(errors))

Number of errors: 2711


In [53]:
# 3 examples of mistakes
error_examples = df.loc[errors.index, [
    "client_id",
    "content_id",
    "content_type",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction"
]].copy()

error_examples["predicted_probability"] = errors["probability"]

error_examples.head(3)

,client_id,content_id,content_type,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction,predicted_probability
13,client_8527a891e2,content_a5a2fbc76336,keyword article,307,0.00,39.8,238,103,stable,0.615
19,client_f369cb89fc,content_af865035b328,keyword article,99,2.02,6.9,187,20,down,0.365
25,client_f369cb89fc,content_033ae3e7aecf,keyword article,27,0.00,7.2,180,20,down,0.395


**Interpretation:**

The model relied most on impressions_90d, avg_position, and content_age_days.
It made 2,711 classification errors (using a 0.5 probability threshold) , showing that some pages were difficult to distinguish using these features alone.
This suggests that the model captures useful patterns but still has difficulty separating some declining and stable pages


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.